# Atlas-warping QA

Sanity-check for `05_masking/warp_atlases_to_T1w.sh` + `make_atlas_region_masks.py`:
before trusting any T1w-space (or MNI-space) ROI stats, confirm here that

1. **the warped atlas lands in the right place** -- overlaid directly on the subject's
   own T1w (or the MNI template), so a wrong-direction transform or a mismatched
   reference image is obvious by eye, not just assumed correct; and
2. **no region got the wrong label** -- each region is plotted individually with its
   name, its integer label value, and its voxel count in the title, so a copy-paste
   error in one of the `roi_dict_*` mappings in `make_atlas_region_masks.py` shows up
   as an obviously-wrong location rather than silently producing a mislabeled ROI.

This directly checks the risk flagged in `REVISION_PLAN.md` Workstream 8: the
MNI->T1w transform direction `warp_atlases_to_T1w.sh` assumes was never verified
against real fMRIPrep 25.2.5 output.

No new plotting code beyond simple loops over `nilearn.plotting.plot_roi` --
mirrors the existing ad hoc "check masks" cell in `masking.ipynb`, generalized to
any subject/space/atlas. Label dictionaries and atlas/mask-directory paths are
imported directly from `make_atlas_region_masks.py` (`resolve_atlas_paths` +
`roi_dict_*`) rather than redefined here, so this notebook can't silently drift
out of sync with what actually generated the masks. The cortical (auditory/PFC)
section also reuses `roi_surface_plotting.plot_roi_surface_stat` for a labeled
fsaverage parcellation view, the same function `group_level_all_ROI.ipynb` uses
for real statistics -- here it's just fed a per-region index instead of a t-stat.

## Configuration

In [ ]:
import os
import sys
from glob import glob

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import plotting, datasets

sys.path.insert(0, '.')       # make_atlas_region_masks.py, this directory
sys.path.insert(0, '..')      # roi_surface_plotting.py, repo root

from make_atlas_region_masks import resolve_atlas_paths
from roi_surface_plotting import plot_roi_surface_stat

In [ ]:
# subject(s) to QA -- start with one, extend the list once this looks right
sub_list_qa = ['FLT02']

bidsroot = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised/'
fmriprep_dir = os.path.join(bidsroot, 'derivatives', 'denoised_fmriprep-25.2.5')
nilearn_dir = os.path.join(bidsroot, 'derivatives', 'nilearn')

# T1w atlases to check (space_label, atlas_label) -- everything warp_atlases_to_T1w.sh
# + make_atlas_region_masks.py currently produce for T1w space
t1w_atlases = [
    ('T1w', 'tian_S2'),
    ('T1w', 'subcort_aud'),
    ('T1w', 'carpet_dseg'),
    ('T1w', 'carpet_pfc'),
]

# the two cortical (surface-projectable) atlases, for the fsaverage parcellation view
cortical_atlases = [('T1w', 'carpet_dseg'), ('T1w', 'carpet_pfc')]

## Background reference image per subject/space

T1w-space atlases get overlaid on that subject's own preprocessed T1w (the same
file `warp_atlases_to_T1w.sh` warps into) so misalignment is visible pixel-for-pixel.
MNI-space atlases get overlaid on the standard template, matching
`group_level_all_ROI.ipynb`'s own convention.

In [ ]:
_mni_template = None  # lazy-loaded, only needed if an MNI-space atlas is checked

def get_bg_img(sub_id, space_label, fmriprep_dir):
    global _mni_template
    if space_label == 'T1w':
        return os.path.join(fmriprep_dir, f'sub-{sub_id}', 'anat',
                            f'sub-{sub_id}_desc-preproc_T1w.nii.gz')
    elif space_label == 'MNI152NLin2009cAsym':
        if _mni_template is None:
            _mni_template = datasets.load_mni152_template(resolution=1)
        return _mni_template
    raise ValueError(f"no background image defined for space_label={space_label!r}")

## Tier 1 -- does the warped atlas land in the right place?

Plots the *whole* multi-label atlas file directly (before it gets split into
per-region masks and resampled onto the functional grid), so this is the most
direct check of the ANTs warp itself: right hemisphere on the right side, right
tissue class, no obvious rotation/flip/scaling artifact.

In [ ]:
def qa_plot_full_atlas(sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir,
                       cut_coords=None, figure=None):
    atlas_fpath, _, roi_dict = resolve_atlas_paths(
        sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir=fmriprep_dir)
    if not os.path.exists(atlas_fpath):
        # nilearn's own missing-file error here is a ValueError, not
        # FileNotFoundError -- raise the type the caller below actually expects
        raise FileNotFoundError(atlas_fpath)
    bg_img = get_bg_img(sub_id, space_label, fmriprep_dir)

    display = plotting.plot_roi(
        atlas_fpath, bg_img=bg_img,
        title=f'sub-{sub_id} {atlas_label} ({space_label}, {len(roi_dict)} regions)',
        cmap='tab20', alpha=0.6, draw_cross=False, black_bg=False,
        cut_coords=cut_coords, figure=figure,
    )
    return display

In [ ]:
for sub_id in sub_list_qa:
    for space_label, atlas_label in t1w_atlases:
        try:
            qa_plot_full_atlas(sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir)
            plotting.show()
        except FileNotFoundError as e:
            print(f'sub-{sub_id} {atlas_label} ({space_label}): not generated yet -- {e}')

## Tier 2 -- per-region masks: confirm labels/values and no mixups

Loops over the *actual* per-region mask files `make_atlas_region_masks.py` writes
(already resampled onto the functional grid, exactly what the real analysis reads),
one panel per region, titled with the region name, its integer atlas label value,
and its voxel count. A region with a suspiciously low/zero voxel count or a panel
that clearly doesn't match its name (e.g. "L-HG" not over left Heschl's gyrus) means
a `roi_dict_*` entry in `make_atlas_region_masks.py` is wrong.

In [ ]:
def _region_mask_fpath(sub_mask_dir, sub_id, space_label, region_name):
    return os.path.join(sub_mask_dir, f'sub-{sub_id}_space-{space_label}_mask-{region_name}.nii.gz')

def qa_plot_region_grid(sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir,
                        n_cols=4, dpi=150):
    _, sub_mask_dir, roi_dict = resolve_atlas_paths(
        sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir=fmriprep_dir)
    bg_img = get_bg_img(sub_id, space_label, fmriprep_dir)

    regions = list(roi_dict.items())
    n_rows = -(-len(regions) // n_cols)  # ceil
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 2.6 * n_rows), dpi=dpi)
    axes = np.atleast_2d(axes)
    fig.suptitle(f'sub-{sub_id} {atlas_label} ({space_label}) -- per-region masks')

    for i, (region_name, label_value) in enumerate(regions):
        ax = axes[i // n_cols][i % n_cols]
        mask_fpath = _region_mask_fpath(sub_mask_dir, sub_id, space_label, region_name)
        if not os.path.exists(mask_fpath):
            ax.set_title(f'{region_name} (id={label_value})\nMISSING', fontsize=8, color='red')
            ax.axis('off')
            continue
        n_voxels = int(nib.load(mask_fpath).get_fdata().sum())
        plotting.plot_roi(
            mask_fpath, bg_img=bg_img, axes=ax, figure=fig,
            title=f'{region_name} (id={label_value}, n={n_voxels})',
            cmap='autumn', alpha=0.8, draw_cross=False, black_bg=False,
            colorbar=False,
        )

    for j in range(len(regions), n_rows * n_cols):
        axes[j // n_cols][j % n_cols].axis('off')

    fig.tight_layout()
    return fig

In [ ]:
for sub_id in sub_list_qa:
    for space_label, atlas_label in t1w_atlases:
        try:
            qa_plot_region_grid(sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir)
            plt.show()
        except FileNotFoundError as e:
            print(f'sub-{sub_id} {atlas_label} ({space_label}): not generated yet -- {e}')

## Cortical parcellation overview (fsaverage surface)

Only applies to the two cortical, surface-projectable atlases (`carpet_dseg` /
auditory regions, `carpet_pfc`) -- reuses `roi_surface_plotting.plot_roi_surface_stat`
unchanged, the same function `group_level_all_ROI.ipynb` uses to plot real
statistics, just fed a per-region index instead of a t-value so every region
renders as a distinct color with its name labeled directly on the brain. Good for
spotting a swapped hemisphere or a region assigned to the wrong gyrus at a glance,
across the whole parcellation at once rather than one region at a time.

In [ ]:
def qa_plot_cortical_parcellation(sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir,
                                  fsaverage=None):
    _, sub_mask_dir, roi_dict = resolve_atlas_paths(
        sub_id, space_label, atlas_label, nilearn_dir, fmriprep_dir=fmriprep_dir)

    region_names = list(roi_dict.keys())
    stat_dict = {region: i for i, region in enumerate(region_names)}
    mask_path_dict = {
        region: _region_mask_fpath(sub_mask_dir, sub_id, space_label, region)
        for region in region_names
    }
    missing = [r for r, p in mask_path_dict.items() if not os.path.exists(p)]
    if missing:
        print(f'sub-{sub_id} {atlas_label}: missing mask files for {missing}, skipping')
        return None

    return plot_roi_surface_stat(
        stat_dict, mask_path_dict, fsaverage=fsaverage,
        cmap='tab20', title=f'sub-{sub_id} {atlas_label} ({space_label}) parcellation',
    )

In [ ]:
fsaverage = datasets.fetch_surf_fsaverage('fsaverage')  # fetch once, reused below

for sub_id in sub_list_qa:
    for space_label, atlas_label in cortical_atlases:
        fig = qa_plot_cortical_parcellation(sub_id, space_label, atlas_label, nilearn_dir,
                                            fmriprep_dir, fsaverage=fsaverage)
        if fig is not None:
            plt.show()

## What to look for

- **Tier 1**: hemispheres on the correct side, subcortical atlases actually inside
  the subcortex (not floating outside the brain -- the classic wrong-direction-
  transform failure mode), no gross rotation/shear relative to the T1w underneath.
- **Tier 2**: every region present (no "MISSING" panels once masks are generated),
  voxel counts in a plausible range for that structure's size, and each panel's
  highlighted location actually matches its name.
- **Surface view**: for `carpet_dseg`/`carpet_pfc`, left/right regions on the
  correct hemisphere and each label sitting on the gyrus/sulcus its name implies
  (e.g. `L-HG` on left Heschl's gyrus, not STG).

Once this looks right for one subject, add more `sub_list_qa` entries and re-run
before trusting group-level T1w-space ROI stats built from these masks.